### Tools

Models can request to call that perform tasks such as fetching data from database,searching the web , or running code.Tools are pairing of:

1. A schema , including the name of the tool , a description ,and /or argument defination(often a JSON schema)
2. A function or coroutine to execute 

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Why do parrort talk?")
response

AIMessage(content="<think>\nOkay, the user is asking why parrots talk. Let me start by recalling what I know about parrots. They're known for their ability to mimic human speech, right? But why exactly do they do that?\n\nFirst, maybe it's a form of communication. In the wild, parrots might mimic sounds to interact with others of their kind. But when they're pets, they might mimic humans instead. But is that the main reason? I should check if there's more to it.\n\nI remember reading that parrots are highly social animals. In their natural habitats, they live in flocks, so vocalizations help them stay in contact. If they're in captivity, they might direct their social behaviors towards humans, so talking is a way to communicate with their human companions.\n\nAnother angle is their intelligence. Parrots are among the most intelligent birds. They might learn to talk as a way to engage with their environment and interact with humans. It could be a form of play or a way to get attention o

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [3]:
response=model_with_tools.invoke("Whats the weather in Boston")
print(response)

for tool_call in response.tool_calls:
    #View tool calls made by models
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the JSON is correctly formatted with the required parameter. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'fnt4afenz', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 150, 'total_tokens': 256, 'completion_time': 0.173587581, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.005819756, 'prompt_tokens_details': None, 'queue_time': 0.049005794, 'total_time': 0.179407337}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 

### Tools Execution Loops

In [4]:
# Step 1: Model generates tool calls

messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The weather in Boston is 72°F and sunny."

The weather in Boston is sunny. If you're there, it's a great day to enjoy outdoor activities! 🌞
